# LeetCode 492 — Construct the Rectangle bằng Python

**Link đề:** https://leetcode.com/problems/construct-the-rectangle/
**Độ khó:** Easy · **Chủ đề:** Math (ước số, căn bậc hai)

```python
class Solution:
    def constructRectangle(self, area: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu đề: tìm hình chữ nhật `L × W = area`, `L >= W`, và `L − W` nhỏ nhất.
2. Ôn phép chia lấy dư `%`, chia nguyên `//`, `range` đi lùi, `math.isqrt` (căn bậc hai nguyên) và vì sao **không nên** dùng `int(math.sqrt(x))` một cách mù quáng.
3. Đi từ vét cạn mọi `W` từ 1 đến `area` → chỉ cần thử `W <= √area` → **bắt đầu từ `⌊√area⌋` đi lùi, gặp ước đầu tiên là dừng**.
4. Chứng minh "ước gần căn nhất cho hiệu nhỏ nhất", trace, phân tích Big-O (trường hợp xấu: `area` là số nguyên tố).
5. Làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Một lập trình viên web cần thiết kế trang hình chữ nhật có **diện tích** cho trước `area`. Tìm chiều dài `L` và chiều rộng `W` sao cho:
1. `L × W == area`.
2. `W <= L` (rộng không lớn hơn dài).
3. `L − W` **nhỏ nhất có thể** (càng gần hình vuông càng tốt).

Trả về `[L, W]`.

| `area` | Các cặp `[L, W]` hợp lệ | Output |
|---:|---|---|
| 4 | `[4,1]`, `[2,2]` | `[2,2]` |
| 37 | `[37,1]` (37 là số nguyên tố) | `[37,1]` |
| 122122 | …, `[427,286]` | `[427,286]` |

**Constraints:** `1 <= area <= 10^7`.

Gợi ý của đề: `W` luôn `<= √area`, nên bắt đầu tìm từ `√area` trở xuống.

In [1]:
area = 12
pairs = [(area // w, w) for w in range(1, area + 1) if area % w == 0 and w <= area // w]
print("Các cặp [L, W] hợp lệ của 12:", pairs)
print("Hiệu L - W:", [L - W for L, W in pairs])

Các cặp [L, W] hợp lệ của 12: [(12, 1), (6, 2), (4, 3)]
Hiệu L - W: [11, 4, 1]


Với `area = 12`: các cặp `[12,1]`, `[6,2]`, `[4,3]` có hiệu 11, 4, 1 → đáp án `[4,3]` — cặp có `W` **lớn nhất** (gần √12 ≈ 3.46 nhất).

Câu hỏi đặt ra trước khi code:
- `W` có thể lớn tới đâu nếu `W <= L` và `L × W = area`?
- Vì sao `W` lớn nhất (trong giới hạn đó) lại cho hiệu nhỏ nhất?
- Tính `√area` thế nào cho an toàn với số nguyên?

## 2. Nền tảng Python và toán cần biết

### 2.1. Ước số với `%` và `//`

`area % w == 0` ⇔ `w` là **ước** của `area`. Khi đó cặp tương ứng là `area // w` (phép chia **nguyên**, kết quả kiểu `int`). Dùng `/` sẽ ra số thực (`float`), ví dụ `12 / 3 == 4.0` — đề yêu cầu trả số nguyên.

In [2]:
print(12 % 3, 12 % 5)       # 0 -> 3 là ước; 2 -> 5 không phải
print(12 // 3, 12 / 3)      # 4 và 4.0
print(type(12 // 3), type(12 / 3))

0 2
4 4.0
<class 'int'> <class 'float'>


### 2.2. Vì sao `W <= √area`?

Nếu `W <= L` thì `W × W <= W × L = area`, nên `W <= √area`. Ngược lại `L >= √area`. Vậy ước "nhỏ" luôn nằm **dưới hoặc bằng** căn, ước "lớn" nằm **trên hoặc bằng** căn — các ước đi thành từng **cặp đối xứng qua √area**.

In [3]:
area = 36
for w in range(1, area + 1):
    if area % w == 0:
        side = "<= √36" if w * w <= area else ">  √36"
        print(f"ước {w:>2} ({side}) bắt cặp với {area // w}")

ước  1 (<= √36) bắt cặp với 36
ước  2 (<= √36) bắt cặp với 18
ước  3 (<= √36) bắt cặp với 12
ước  4 (<= √36) bắt cặp với 9
ước  6 (<= √36) bắt cặp với 6
ước  9 (>  √36) bắt cặp với 4
ước 12 (>  √36) bắt cặp với 3
ước 18 (>  √36) bắt cặp với 2
ước 36 (>  √36) bắt cặp với 1


### 2.3. `math.isqrt` — căn bậc hai **nguyên**

`math.isqrt(n)` trả `⌊√n⌋` (số nguyên lớn nhất có bình phương `<= n`), tính hoàn toàn bằng số nguyên → **luôn chính xác**.

`int(math.sqrt(n))` tính qua số thực `float` (độ chính xác ~15–16 chữ số) — với `n <= 10^7` thì vẫn đúng, nhưng với số rất lớn có thể sai 1 đơn vị. Thói quen tốt: dùng `isqrt` cho bài số nguyên.

In [4]:
import math
for n in [1, 4, 12, 37, 122122, 10**7]:
    print(f"n={n:>9}: math.sqrt={math.sqrt(n):.4f}, isqrt={math.isqrt(n)}")

big = (10**8 + 1) ** 2 - 1           # √big hơi nhỏ hơn 10^8 + 1
print("int(math.sqrt(big)) =", int(math.sqrt(big)))
print("math.isqrt(big)     =", math.isqrt(big))
print("đúng là 10**8       =", 10**8)

n=        1: math.sqrt=1.0000, isqrt=1
n=        4: math.sqrt=2.0000, isqrt=2
n=       12: math.sqrt=3.4641, isqrt=3
n=       37: math.sqrt=6.0828, isqrt=6
n=   122122: math.sqrt=349.4596, isqrt=349
n= 10000000: math.sqrt=3162.2777, isqrt=3162
int(math.sqrt(big)) = 100000001
math.isqrt(big)     = 100000000
đúng là 10**8       = 100000000


Với `big ≈ 10^16`, `math.sqrt` trả số thực bị làm tròn lên thành đúng `10^8 + 1` (sai), còn `isqrt` cho `10^8` (đúng). Bài này `area <= 10^7` nên chưa gặp, nhưng biết để tránh bẫy ở bài khác.

### 2.4. `range` đi lùi

`range(start, stop, -1)` đếm lùi từ `start` xuống **trên** `stop` (không gồm `stop`). Muốn đi tới 1 thì `stop = 0`.

In [5]:
print(list(range(5, 0, -1)))     # 5 4 3 2 1
print(list(range(3, 0, -1)))

[5, 4, 3, 2, 1]
[3, 2, 1]


## 3. Cách 1 — Vét cạn: thử mọi `W` từ 1 đến `area`

Duyệt `W = 1..area`; nếu `W` là ước và `W <= area // W` thì cập nhật đáp án nếu hiệu nhỏ hơn. `O(area)` phép chia — với `area = 10^7` là 10 triệu vòng Python (vài giây) → chậm.

In [6]:
def rectangle_bruteforce(area: int) -> list[int]:
    best = None
    for w in range(1, area + 1):
        if area % w == 0:
            L = area // w
            if L >= w and (best is None or L - w < best[0] - best[1]):
                best = [L, w]
    return best

print(rectangle_bruteforce(4), rectangle_bruteforce(37), rectangle_bruteforce(122122))

[2, 2] [37, 1] [427, 286]


`best is None`: `None` là giá trị "chưa có gì"; so sánh với `None` dùng `is` (kiểm tra **đúng đối tượng**) chứ không dùng `==`.

## 4. Cách 2 — Chỉ thử `W` từ 1 đến `⌊√area⌋`

Vì `W <= √area`, ta bỏ được hơn 99.9% vòng lặp: với `area = 10^7`, chỉ còn ~3162 vòng. Và vì hiệu `L − W` **giảm** khi `W` tăng (mục 5), cặp cuối cùng tìm thấy chính là đáp án → không cần so sánh hiệu.

In [7]:
def rectangle_up_to_sqrt(area: int) -> list[int]:
    answer = [area, 1]
    for w in range(1, math.isqrt(area) + 1):
        if area % w == 0:
            answer = [area // w, w]      # W càng lớn càng tốt -> ghi đè
    return answer

print(rectangle_up_to_sqrt(4), rectangle_up_to_sqrt(37), rectangle_up_to_sqrt(122122))

[2, 2] [37, 1] [427, 286]


## 5. Bước đột phá — hiệu `L − W` giảm khi `W` tiến về √area

Với `W` là ước và `W <= √area`: `L − W = area / W − W`. Hàm `f(W) = area/W − W` **giảm ngặt** khi `W` tăng (tử `area/W` giảm, `−W` cũng giảm). Vậy:

> Ước **lớn nhất** không vượt quá `√area` cho hiệu nhỏ nhất.

Do đó tìm **từ trên xuống**: bắt đầu `W = ⌊√area⌋`, giảm dần; **ước đầu tiên** gặp được là đáp án → dừng ngay. Luôn dừng vì `W = 1` chắc chắn là ước.

In [8]:
area = 122122
print("f(W) = area//W - W cho vài ước W <= √area:")
for w in range(1, math.isqrt(area) + 1):
    if area % w == 0:
        print(f"  W={w:>3}, L={area // w:>6}, L - W = {area // w - w}")

f(W) = area//W - W cho vài ước W <= √area:
  W=  1, L=122122, L - W = 122121
  W=  2, L= 61061, L - W = 61059
  W=  7, L= 17446, L - W = 17439
  W= 11, L= 11102, L - W = 11091
  W= 13, L=  9394, L - W = 9381
  W= 14, L=  8723, L - W = 8709
  W= 22, L=  5551, L - W = 5529
  W= 26, L=  4697, L - W = 4671
  W= 61, L=  2002, L - W = 1941
  W= 77, L=  1586, L - W = 1509
  W= 91, L=  1342, L - W = 1251
  W=122, L=  1001, L - W = 879
  W=143, L=   854, L - W = 711
  W=154, L=   793, L - W = 639
  W=182, L=   671, L - W = 489
  W=286, L=   427, L - W = 141


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép

### 6.1. Điểm xuất phát
```python
w = math.isqrt(area)        # W lớn nhất có thể
```
### 6.2. Đi lùi đến khi gặp ước
```python
while area % w != 0:
    w -= 1
```
### 6.3. Trả về `[L, W]` (L trước!)
```python
return [area // w, w]
```

In [9]:
def rectangle_from_sqrt(area: int) -> list[int]:
    w = math.isqrt(area)
    while area % w != 0:
        w -= 1
    return [area // w, w]

for a in [4, 37, 122122, 1, 12, 10**7]:
    print(a, "->", rectangle_from_sqrt(a))

4 -> [2, 2]
37 -> [37, 1]
122122 -> [427, 286]
1 -> [1, 1]
12 -> [4, 3]
10000000 -> [3200, 3125]


## 7. TRACE từng bước

### 7.1. `area = 12`: √12 ≈ 3.46 → `isqrt = 3`

| `w` | `12 % w` | ước? | hành động |
|---|---|---|---|
| 3 | 0 | ✔ | dừng → `[12 // 3, 3] = [4, 3]` |

### 7.2. `area = 37` (số nguyên tố): `isqrt = 6`

| `w` | `37 % w` | ước? |
|---|---|---|
| 6 | 1 | ✘ |
| 5 | 2 | ✘ |
| 4 | 1 | ✘ |
| 3 | 1 | ✘ |
| 2 | 1 | ✘ |
| 1 | 0 | ✔ → `[37, 1]` |

Số nguyên tố là **trường hợp xấu nhất**: phải đi lùi hết tới 1.

In [10]:
def trace_rectangle(area):
    w = math.isqrt(area)
    print(f"area={area}, √area≈{math.sqrt(area):.3f}, bắt đầu w = isqrt = {w}")
    steps = 1
    while area % w != 0:
        if steps <= 4:
            print(f"  w={w}: {area} % {w} = {area % w} -> không phải ước, giảm w")
        elif steps == 5:
            print("  ... (bỏ bớt các dòng giống nhau)")
        w -= 1
        steps += 1
    print(f"  w={w}: {area} % {w} = 0 -> ước! kết quả [L, W] = [{area // w}, {w}] sau {steps} lần thử")
    return [area // w, w]

trace_rectangle(12)
trace_rectangle(37)
trace_rectangle(122122)

area=12, √area≈3.464, bắt đầu w = isqrt = 3
  w=3: 12 % 3 = 0 -> ước! kết quả [L, W] = [4, 3] sau 1 lần thử
area=37, √area≈6.083, bắt đầu w = isqrt = 6
  w=6: 37 % 6 = 1 -> không phải ước, giảm w
  w=5: 37 % 5 = 2 -> không phải ước, giảm w
  w=4: 37 % 4 = 1 -> không phải ước, giảm w
  w=3: 37 % 3 = 1 -> không phải ước, giảm w
  ... (bỏ bớt các dòng giống nhau)
  w=1: 37 % 1 = 0 -> ước! kết quả [L, W] = [37, 1] sau 6 lần thử
area=122122, √area≈349.460, bắt đầu w = isqrt = 349
  w=349: 122122 % 349 = 321 -> không phải ước, giảm w
  w=348: 122122 % 348 = 322 -> không phải ước, giảm w
  w=347: 122122 % 347 = 325 -> không phải ước, giảm w
  w=346: 122122 % 346 = 330 -> không phải ước, giảm w
  ... (bỏ bớt các dòng giống nhau)
  w=286: 122122 % 286 = 0 -> ước! kết quả [L, W] = [427, 286] sau 64 lần thử


[427, 286]

## 8. Cách 4 (đối xứng) — tìm `L` nhỏ nhất `>= √area` đi lên

Cũng có thể tìm từ phía `L`: `L` bắt đầu từ `⌈√area⌉` và **tăng** dần đến khi `L` là ước. Cách này đúng nhưng thường chậm hơn: trong trường hợp xấu (số nguyên tố) `L` phải đi từ √area lên tới `area` — tức `O(area)` thay vì `O(√area)`. Đây là ví dụ hay: **cùng ý tưởng, chọn sai phía thì độ phức tạp tệ hẳn**.

Làm tròn **lên** căn bằng số nguyên: `r = isqrt(n)`, nếu `r * r < n` thì `r += 1` (khi đó `r = ⌈√n⌉`).

In [11]:
def rectangle_from_L(area: int) -> tuple[list[int], int]:   # trả thêm số bước để so sánh
    L = math.isqrt(area)
    if L * L < area:
        L += 1                       # ⌈√area⌉
    steps = 0
    while area % L != 0:
        L += 1
        steps += 1
    return [L, area // L], steps

for a in [12, 122122, 37, 9_999_991]:     # 9_999_991 là số nguyên tố
    (res, steps) = rectangle_from_L(a)
    w = math.isqrt(a); s2 = 0
    while a % w: w -= 1; s2 += 1
    print(f"area={a:>8}: {res}, đi lên L mất {steps:>8} bước | đi xuống W mất {s2:>5} bước")

area=      12: [4, 3], đi lên L mất        0 bước | đi xuống W mất     0 bước
area=  122122: [427, 286], đi lên L mất       77 bước | đi xuống W mất    63 bước
area=      37: [37, 1], đi lên L mất       30 bước | đi xuống W mất     5 bước


area= 9999991: [9999991, 1], đi lên L mất  9996828 bước | đi xuống W mất  3161 bước


## 9. Code nộp LeetCode (Python 3)

In [12]:
import math

class Solution:
    def constructRectangle(self, area: int) -> list[int]:
        w = math.isqrt(area)
        while area % w != 0:
            w -= 1
        return [area // w, w]

sol = Solution()
print(sol.constructRectangle(4))        # [2, 2]
print(sol.constructRectangle(37))       # [37, 1]
print(sol.constructRectangle(122122))   # [427, 286]

[2, 2]
[37, 1]
[427, 286]


## 10. Vì sao đúng?

1. **Hợp lệ:** vòng lặp dừng tại `w` thỏa `area % w == 0`, nên `L = area // w` là số nguyên và `L × w = area`. Vì `w <= ⌊√area⌋ <= √area` nên `w² <= area = L·w` → `w <= L` ✔.
2. **Dừng chắc chắn:** `w` giảm từ `⌊√area⌋ >= 1` và `area % 1 == 0` → không bao giờ xuống dưới 1 (không chia cho 0).
3. **Tối ưu:** mọi cặp hợp lệ `[L', W']` có `W' <= √area`, tức `W' <= ⌊√area⌋`. Vòng lặp duyệt các giá trị từ `⌊√area⌋` xuống và dừng ở **ước đầu tiên** → `w` là ước lớn nhất `<= √area`, nên `W' <= w`. Vì `f(W) = area/W − W` giảm ngặt, `L' − W' = f(W') >= f(w) = L − w` ✔.

## 11. Phân tích Time / Space

Gọi `A = area`.

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `W = 1..A` | `O(A)` | `O(1)` | ~10^7 vòng: chậm |
| 2. `W = 1..√A`, giữ ước cuối | `Θ(√A)` luôn | `O(1)` | luôn đi hết |
| 3. `W` từ `√A` đi xuống (**chính**) | `O(√A)` xấu nhất, thường rất nhanh | `O(1)` | dừng ở ước đầu tiên |
| 4. `L` từ `√A` đi lên | `O(A)` xấu nhất | `O(1)` | chọn sai phía |

Với `A <= 10^7`: `√A <= 3163` vòng — rất nhanh. `math.isqrt` chạy `O(log A)`-ish, không đáng kể.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", 4, [2, 2]),
    ("đề 2 (số nguyên tố)", 37, [37, 1]),
    ("đề 3", 122122, [427, 286]),
    ("area = 1", 1, [1, 1]),
    ("area = 2", 2, [2, 1]),
    ("số chính phương", 10**6, [1000, 1000]),
    ("12", 12, [4, 3]),
    ("max 10^7", 10**7, [3200, 3125]),
    ("số nguyên tố lớn", 9_999_991, [9_999_991, 1]),
]
sol = Solution()
for name, a, expected in cases:
    check(name, sol.constructRectangle(a), expected)

PASS  đề 1: [2, 2]
PASS  đề 2 (số nguyên tố): [37, 1]
PASS  đề 3: [427, 286]
PASS  area = 1: [1, 1]
PASS  area = 2: [2, 1]
PASS  số chính phương: [1000, 1000]
PASS  12: [4, 3]
PASS  max 10^7: [3200, 3125]
PASS  số nguyên tố lớn: [9999991, 1]


In [14]:
import random
random.seed(492)
for a in list(range(1, 3001)) + [random.randint(1, 10**6) for _ in range(30)]:
    expected = rectangle_up_to_sqrt(a)
    got = Solution().constructRectangle(a)
    assert got == expected == rectangle_from_L(a)[0], (a, got, expected)
    L, W = got
    assert L * W == a and L >= W
for a in range(1, 400):
    assert rectangle_bruteforce(a) == Solution().constructRectangle(a)
print("1..3000 và 30 số ngẫu nhiên: các cách khớp nhau; 1..399 khớp vét cạn")

1..3000 và 30 số ngẫu nhiên: các cách khớp nhau; 1..399 khớp vét cạn


## 13. Lỗi người mới hay mắc

### Lỗi 1: trả về `[W, L]` thay vì `[L, W]`
Đề yêu cầu `L` trước và `L >= W`.

In [15]:
def wrong_order(area):
    w = math.isqrt(area)
    while area % w:
        w -= 1
    return [w, area // w]           # SAI thứ tự

print("sai :", wrong_order(122122))
print("đúng:", rectangle_from_sqrt(122122))

sai : [286, 427]
đúng: [427, 286]


### Lỗi 2: dùng `/` thay vì `//`
Kết quả `[427.0, 286]` có `float` — LeetCode yêu cầu `list[int]`, và với số lớn `/` có thể mất chính xác.

In [16]:
print("sai :", [122122 / 286, 286])
print("đúng:", [122122 // 286, 286])

sai : [427.0, 286]
đúng: [427, 286]


### Lỗi 3: xuất phát từ `math.ceil(math.sqrt(area))` (làm tròn **lên**)
Điểm xuất phát lớn hơn √area. Nếu nó lại là ước thì ta dừng ngay với `W > L` — vi phạm điều kiện 2. Thay vì đoán, hãy để máy **tìm phản ví dụ**:

In [17]:
def wrong_ceil(area):
    w = math.ceil(math.sqrt(area))
    while area % w:
        w -= 1
    return [area // w, w]

bad = [a for a in range(1, 200) if wrong_ceil(a) != rectangle_from_sqrt(a)]
print("Các area mà ceil() cho kết quả sai:", bad[:10])
a = bad[0]
print(f"ví dụ area={a}: ceil -> {wrong_ceil(a)} (W > L!) | đúng -> {rectangle_from_sqrt(a)}")

Các area mà ceil() cho kết quả sai: [2, 6, 12, 20, 30, 42, 56, 72, 90, 110]
ví dụ area=2: ceil -> [1, 2] (W > L!) | đúng -> [2, 1]


Còn `round(math.sqrt(area))` thì sao? Thử tương tự trên một miền lớn: không tìm thấy phản ví dụ. Lý do: nếu `w = round(√area) > √area` là ước và `area // w < w` thì `area <= w(w-1) < (w - 0.5)²`, tức `√area < w - 0.5` — khi đó `round` đã phải cho `w - 1`, mâu thuẫn. Dù vậy `isqrt` vẫn là lựa chọn **rõ ràng nhất**: đúng theo định nghĩa, không cần lập luận phụ, không dính `float`.

In [18]:
def try_round(area):
    w = round(math.sqrt(area))
    while area % w:
        w -= 1
    return [area // w, w]

print("round() sai ở:", [a for a in range(1, 20001) if try_round(a) != rectangle_from_sqrt(a)])

round() sai ở: []


### Lỗi 4: đi lùi tới 0 → `ZeroDivisionError`
Nếu viết `while w > 0 and area % w != 0` rồi lỡ khởi tạo sai, hoặc dùng `range(isqrt, 0, -1)` mà quên `return` trong vòng lặp, có thể gặp chia cho 0 hoặc trả `None`. Với khởi tạo `w = isqrt(area) >= 1` và điều kiện `area % w != 0`, vòng lặp luôn dừng ở `w >= 1`.

### Lỗi 5: đi **lên** từ 1 và dừng ở ước đầu tiên
Ước đầu tiên từ dưới lên luôn là `1` → trả `[area, 1]` — hiệu **lớn nhất**, sai hoàn toàn (trừ khi `area` nguyên tố).

In [19]:
def wrong_from_one(area):
    for w in range(1, area + 1):
        if area % w == 0:
            return [area // w, w]

print("sai :", wrong_from_one(12), "| đúng:", rectangle_from_sqrt(12))

sai : [12, 1] | đúng: [4, 3]


## 14. Mẹo nhận dạng pattern

- "Tích cố định, muốn hai thừa số **gần nhau nhất**" → thử ước từ `√n` đi ra ngoài.
- "Duyệt ước số" → chỉ cần `i` từ 1 đến `√n`, mỗi `i` cho cặp `(i, n // i)` → `O(√n)`.
- "Căn bậc hai cho số nguyên" → `math.isqrt`, tránh `float`.
- Tối ưu hóa "tìm cực trị": nếu biết hàm mục tiêu **đơn điệu** theo hướng duyệt, dừng ngay ở phần tử hợp lệ đầu tiên.

Bài cùng họ: 367 Valid Perfect Square, 69 Sqrt(x), 1492 The kth Factor of n, 507 Perfect Number, 1362 Closest Divisors, 633 Sum of Square Numbers.

## 15. Biến thể

### 15.1. Liệt kê mọi ước trong `O(√n)`

In [20]:
def divisors(n: int) -> list[int]:
    small, large = [], []
    for i in range(1, math.isqrt(n) + 1):
        if n % i == 0:
            small.append(i)
            if i != n // i:              # tránh thêm căn hai lần khi n chính phương
                large.append(n // i)
    return small + large[::-1]

print(divisors(36))
print(divisors(37))
print(divisors(122122))

[1, 2, 3, 4, 6, 9, 12, 18, 36]
[1, 37]
[1, 2, 7, 11, 13, 14, 22, 26, 61, 77, 91, 122, 143, 154, 182, 286, 427, 671, 793, 854, 1001, 1342, 1586, 2002, 4697, 5551, 8723, 9394, 11102, 17446, 61061, 122122]


### 15.2. Closest Divisors (LeetCode 1362)
Cho `num`, tìm hai số có tích là `num + 1` **hoặc** `num + 2` và hiệu tuyệt đối nhỏ nhất — áp dụng đúng hàm của bài này hai lần rồi chọn cặp tốt hơn.

In [21]:
def closest_divisors(num: int) -> list[int]:
    c1 = rectangle_from_sqrt(num + 1)
    c2 = rectangle_from_sqrt(num + 2)
    return c1 if c1[0] - c1[1] <= c2[0] - c2[1] else c2

print(closest_divisors(8), closest_divisors(123), closest_divisors(999))

[3, 3] [25, 5] [40, 25]


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Tính tay
`area = 50`. `isqrt(50)` là bao nhiêu? Lập bảng đi lùi đến ước đầu tiên; đáp án `[L, W]`?

### Bài 2 — Hoàn thiện lời giải chính

In [22]:
def ex2_construct(area: int) -> list[int]:
    # TODO 1: w = math.isqrt(area)
    # TODO 2: trong khi w không phải ước, giảm w
    # TODO 3: trả [area // w, w]
    pass

# assert ex2_construct(4) == [2, 2]
# assert ex2_construct(37) == [37, 1]

### Bài 3 — Valid Perfect Square (LeetCode 367) không dùng `sqrt`/`isqrt`
Trả `True` nếu `num` là số chính phương. Dùng tìm kiếm nhị phân trên `[1, num]`.

In [23]:
def ex3_is_perfect_square(num: int) -> bool:
    # TODO: binary search lo, hi
    pass

# assert ex3_is_perfect_square(16) is True
# assert ex3_is_perfect_square(14) is False
# assert ex3_is_perfect_square(1) is True

### Bài 4 — Ước thứ `k` (LeetCode 1492)
Trả ước thứ `k` (theo thứ tự tăng dần) của `n`, hoặc `-1` nếu không đủ `k` ước. Ví dụ `n = 12, k = 3` → `3` (các ước 1,2,3,4,6,12).

In [24]:
def ex4_kth_factor(n: int, k: int) -> int:
    # TODO
    pass

# assert ex4_kth_factor(12, 3) == 3
# assert ex4_kth_factor(7, 2) == 7
# assert ex4_kth_factor(4, 4) == -1

### Bài 5 — Mini challenge: hình hộp gần lập phương nhất
Cho `volume`, tìm `[a, b, c]` với `a >= b >= c >= 1`, `a·b·c = volume` và `a − c` nhỏ nhất (nếu hòa, chọn `a` nhỏ nhất). Ví dụ `volume = 8` → `[2, 2, 2]`, `volume = 12` → `[3, 2, 2]`, `volume = 7` → `[7, 1, 1]`.

In [25]:
def ex5_closest_box(volume: int) -> list[int]:
    # TODO: thử c là ước <= căn bậc ba, rồi dùng hàm 2 chiều cho volume // c
    pass

# assert ex5_closest_box(8) == [2, 2, 2]
# assert ex5_closest_box(12) == [3, 2, 2]
# assert ex5_closest_box(7) == [7, 1, 1]

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: `7² = 49 <= 50 < 64`. Mức 2: `50 % 7 = 1`, `50 % 6 = 2`. Mức 3: `w = 5` là ước.

**Bài 2** — Mức 1: chép mục 6. Mức 2: `while area % w != 0: w -= 1`. Mức 3: L trước, W sau.

**Bài 3** — Mức 1: `lo, hi = 1, num`. Mức 2: `mid = (lo + hi) // 2`, so `mid * mid` với `num`. Mức 3: bằng → `True`; nhỏ hơn → `lo = mid + 1`; lớn hơn → `hi = mid - 1`.

**Bài 4** — Mức 1: duyệt `i = 1..n` là đủ với n nhỏ. Mức 2: đếm ngược `k` mỗi khi gặp ước. Mức 3 (nhanh hơn): dùng hàm `divisors` ở 15.1 rồi lấy phần tử `k - 1`.

**Bài 5** — Mức 1: `c` là cạnh nhỏ nhất nên `c³ <= volume`. Mức 2: với mỗi ước `c`, cặp `[a, b]` tốt nhất của `volume // c` có thể tìm bằng hàm bài chính, nhưng phải bảo đảm `b >= c`. Mức 3: thử mọi ước `b` của `rest = volume // c` với `c <= b <= √rest`, lấy cặp có `a − c` nhỏ nhất.

## 17. Đáp án tham khảo

**Bài 1:** `isqrt(50) = 7`; `50 % 7 = 1`, `50 % 6 = 2`, `50 % 5 = 0` → `[10, 5]`.

In [26]:
def ans2_construct(area):
    w = math.isqrt(area)
    while area % w != 0:
        w -= 1
    return [area // w, w]

def ans3_is_perfect_square(num):
    lo, hi = 1, num
    while lo <= hi:
        mid = (lo + hi) // 2
        sq = mid * mid
        if sq == num:
            return True
        if sq < num:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

def ans4_kth_factor(n, k):
    ds = divisors(n)
    return ds[k - 1] if k <= len(ds) else -1

def ans5_closest_box(volume):
    best = None
    c = 1
    while c * c * c <= volume:
        if volume % c == 0:
            rest = volume // c
            for b2 in range(c, math.isqrt(rest) + 1):
                if rest % b2 == 0:
                    cand = [rest // b2, b2, c]
                    key = (cand[0] - cand[2], cand[0])
                    if best is None or key < (best[0] - best[2], best[0]):
                        best = cand
        c += 1
    return best

def brute_box(volume):
    best = None
    for c in range(1, volume + 1):
        for b2 in range(c, volume + 1):
            if volume % (b2 * c) == 0:
                a = volume // (b2 * c)
                if a >= b2:
                    key = (a - c, a)
                    if best is None or key < (best[0] - best[2], best[0]):
                        best = [a, b2, c]
    return best

assert ans2_construct(50) == [10, 5]                                  # bài 1
assert ans2_construct(4) == [2, 2] and ans2_construct(37) == [37, 1]
assert ans3_is_perfect_square(16) and not ans3_is_perfect_square(14) and ans3_is_perfect_square(1)
assert all(ans3_is_perfect_square(x) == (math.isqrt(x) ** 2 == x) for x in range(1, 2000))
assert ans4_kth_factor(12, 3) == 3 and ans4_kth_factor(7, 2) == 7 and ans4_kth_factor(4, 4) == -1
assert ans5_closest_box(8) == [2, 2, 2]
assert ans5_closest_box(12) == [3, 2, 2]
assert ans5_closest_box(7) == [7, 1, 1]
assert all(ans5_closest_box(v) == brute_box(v) for v in range(1, 121))
print("Đáp án bài 1–5: PASS (bài 3 và 5 đối chiếu vét cạn)")

Đáp án bài 1–5: PASS (bài 3 và 5 đối chiếu vét cạn)


So sánh tuple `key < other_key`: Python so **từng phần tử từ trái sang** (thứ tự từ điển) — so `a − c` trước, nếu bằng mới so `a`. Đây là mẹo gọn để xử lý "tiêu chí chính + tiêu chí phụ khi hòa".

## 18. Template nhớ nhanh + checklist submit

```python
import math
w = math.isqrt(area)          # W lớn nhất có thể
while area % w != 0:          # đi lùi đến ước đầu tiên
    w -= 1
return [area // w, w]         # L trước, W sau
```

**Checklist trước khi submit**
- [ ] Trả `[L, W]` với `L >= W` (L đứng trước)?
- [ ] Dùng `//` (số nguyên), không dùng `/`?
- [ ] Điểm xuất phát là `isqrt` (không vượt √area), không phải `round`?
- [ ] Ca `area = 1` → `[1, 1]`; số nguyên tố → `[area, 1]`; số chính phương → `[r, r]`?
- [ ] Giải thích được vì sao ước gần căn nhất cho hiệu nhỏ nhất?